In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/python-dataset-for-dsa-questions/cleaned_flytec.csv
/kaggle/input/taco-cleaned/taco_cleaned.csv
/kaggle/input/cleaned-datasets-of-alpaca/new version.csv
/kaggle/input/leetcode-with-desciption-and-code/leetcode_with_description_and_code.csv


In [2]:
df2 = pd.read_csv("/kaggle/input/cleaned-datasets-of-alpaca/new version.csv")
df3 = pd.read_csv("/kaggle/input/python-dataset-for-dsa-questions/cleaned_flytec.csv")

In [3]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11926 entries, 0 to 11925
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Unnamed: 0          11926 non-null  int64 
 1   instruction         11926 non-null  object
 2   input               8757 non-null   object
 3   output              11926 non-null  object
 4   prompt              11926 non-null  object
 5   classification_raw  11926 non-null  object
 6   category            11926 non-null  object
dtypes: int64(1), object(6)
memory usage: 652.3+ KB


In [4]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  9617 non-null   object
 1   code         9617 non-null   object
 2   category     9617 non-null   object
dtypes: object(3)
memory usage: 225.5+ KB


In [5]:
df3 = df3[['instruction','code']]

In [6]:
df2.head()

,Unnamed: 0,instruction,input,output,prompt,classification_raw,category
0,0,Create a function to calculate the sum of a se...,"[1, 2, 3, 4, 5]",# Python code\ndef sum_sequence(sequence):\n ...,Below is an instruction that describes a task....,Sorting,Sorting
1,3,Generate a python script to perform this action.,"Given a string, remove all the consecutive dup...",def remove_duplicates(string): \n result = ...,Below is an instruction that describes a task....,Sorting,Sorting
2,4,Write a python script to generates random numb...,NaN,def generate_random_divisible_number():\n i...,Below is an instruction that describes a task....,Sorting,Sorting
3,5,Write a Python code to get the third largest e...,"[12, 13, 13, 45, 22, 99]",def third_largest(lst):\n if len(lst) < 3:\...,Below is an instruction that describes a task....,Sorting,Sorting
4,7,Create a Python function that takes in a strin...,"'This is a test', ['test', 'this', 'is']","def contains_words(input_string, words):\n for...",Below is an instruction that describes a task....,searching,searching


In [7]:
df2 = df2[['instruction','output']]

In [8]:
print(df2.info(),df3.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11926 entries, 0 to 11925
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  11926 non-null  object
 1   output       11926 non-null  object
dtypes: object(2)
memory usage: 186.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9617 entries, 0 to 9616
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  9617 non-null   object
 1   code         9617 non-null   object
dtypes: object(2)
memory usage: 150.4+ KB
None None


In [9]:
df2 = df2.rename(columns={'instruction': 'instruction', 'output': 'code'})

In [10]:
combined_df = pd.concat([df2, df3], ignore_index=True)

In [11]:
combined_df.head()

,instruction,code
0,Create a function to calculate the sum of a se...,# Python code\ndef sum_sequence(sequence):\n ...
1,Generate a python script to perform this action.,def remove_duplicates(string): \n result = ...
2,Write a python script to generates random numb...,def generate_random_divisible_number():\n i...
3,Write a Python code to get the third largest e...,def third_largest(lst):\n if len(lst) < 3:\...
4,Create a Python function that takes in a strin...,"def contains_words(input_string, words):\n for..."


In [12]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21543 entries, 0 to 21542
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  21543 non-null  object
 1   code         21543 non-null  object
dtypes: object(2)
memory usage: 336.7+ KB


In [13]:
def complexity_filter(df, text_col='code', min_words=7, max_repeat_ratio=0.4):
    df = df[df[text_col].str.split().str.len() > min_words]
    def repetition_ratio(text):
        words = text.split()
        return 1.0 - (len(set(words)) / len(words)) if len(words) > 0 else 0
    df = df[df[text_col].apply(repetition_ratio) < max_repeat_ratio]
    return df

combined_df = complexity_filter(combined_df)


In [14]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18487 entries, 0 to 21542
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  18487 non-null  object
 1   code         18487 non-null  object
dtypes: object(2)
memory usage: 433.3+ KB


In [15]:
import re
def remove_comments(code):
    return re.sub(r'#.*?\n', '\n', code)
combined_df["code"] = combined_df["code"].apply(remove_comments)
print("Comments removed successfully.")

Comments removed successfully.


In [16]:
import re

def remove_comments_and_docstrings(code):

    # Remove single-line comments
    code = re.sub(r'#.*?\n', '\n', code)

    # Remove triple double-quote docstrings """ """
    code = re.sub(r'""".*?"""', '', code, flags=re.DOTALL)

    # Remove triple single-quote docstrings ''' '''
    code = re.sub(r"'''.*?'''", '', code, flags=re.DOTALL)

    return code


combined_df["code"] = combined_df["code"].apply(remove_comments_and_docstrings)

print("Comments and docstrings removed successfully.")

Comments and docstrings removed successfully.


In [17]:
def filter_only_functions(df, text_col='code'):
    """
    Keep only rows where the text column contains a Python function (starts with 'def').
    """
    mask = df[text_col].str.lower().str.contains(r'\bdef\b')
    return df[mask]

In [18]:
combined_df = filter_only_functions(combined_df, text_col='code')

In [19]:
def normalize_code_blocks(df, text_col='code'):
    def wrap_code(text):
        if not isinstance(text, str):
            return text
        
        text = text.strip()
        
        # If already starts with ```python, leave it
        if text.startswith("```python"):
            return text
        
        # If starts with ``` but without python, convert to ```python
        if text.startswith("```"):
            # Remove first ```
            text = text[3:].strip()
            return f"```python\n{text}\n```"
        
        # Otherwise wrap it
        return f"```python\n{text}\n```"
    
    df[text_col] = df[text_col].apply(wrap_code)
    return df
combined_df = normalize_code_blocks(combined_df)

In [20]:
shuffled_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [21]:
shuffled_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13914 entries, 0 to 13913
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  13914 non-null  object
 1   code         13914 non-null  object
dtypes: object(2)
memory usage: 217.5+ KB


In [22]:
shuffled_df.head(5)

,instruction,code
0,How to use type annotations?,"```python\n\ndef add(a: int, b: int) -> int:\n..."
1,Create a Python program to take an array of nu...,```python\ndef average(nums):\n sum = 0\n ...
2,Create a Python class to implement a multiplic...,```python\nclass MultiplicationTable:\n def...
3,Write a python program that finds a maximum su...,```python\ndef max_subarray(numbers): \n ma...
4,Create a basic Python program to generate a st...,```python\nimport random\nimport string\n\ndef...


In [23]:
train = shuffled_df.iloc[:12522]

In [24]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12522 entries, 0 to 12521
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  12522 non-null  object
 1   code         12522 non-null  object
dtypes: object(2)
memory usage: 195.8+ KB


In [25]:
test = shuffled_df.iloc[12522:]

In [26]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1392 entries, 12522 to 13913
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  1392 non-null   object
 1   code         1392 non-null   object
dtypes: object(2)
memory usage: 21.9+ KB


In [27]:
import json

with open("refined_train.jsonl", "w", encoding="utf-8") as f:
    for _, row in train.iterrows():
        prompt = f"<s>[INST] {row['instruction']} [/INST]"
        code = f"{row['code']} </s>"
        obj = {
            "prompt": prompt,
            "code": code
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("JSONL file created: train.jsonl")

JSONL file created: train.jsonl


In [28]:
import json

with open("refined_test.jsonl", "w", encoding="utf-8") as f:
    for _, row in test.iterrows():
        prompt = f"<s>[INST] {row['instruction']} [/INST]"
        code = f"{row['code']} </s>"
        obj = {
            "prompt": prompt,
            "code": code
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("JSONL file created: test.jsonl")

JSONL file created: test.jsonl
